In [ ]:
%%capture
!pip install tensorboard torchvision
!pip install --upgrade transformers 
!pip install --no-deps trl==0.22.2
!pip install Kernels datasets accelerate evaluate bitsandbytes trl peft protobuf pillow sentencepiece
!pip install huggingface_hub

# Realizando o Login

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Hugging-Face-Token")

In [ ]:
from huggingface_hub import login

login(secret_value_0)

# Carregando o Dataset

In [ ]:
system_think_instruction = """<|think|>
Siga essa linha de pensamento para a análise das imagens de Raio-x de Toráx (PA/Perfil).

1. Ossos e Partes Moles: Checar fraturas em costelas, clavículas e coluna; avaliar cúpulas diafragmáticas e buscar enfisema subcutâneo ou sombras mamárias.
2. Vias Aéreas: Avaliar se a traqueia está centralizada ou desviada e a perviedade dos brônquios principais.
3. Pulmões e Pleura: Buscar opacidades (nódulos, massas, consolidações) ou pneumotórax; checar se os seios costofrênicos estão livres ou velados (derrame).
4. Coração e Mediastino: Meça explicitamente o ICT. Avaliar se há cardiomegalia (ICT > 50%) e checar a anatomia do mediastino, botão aórtico e hilos.
5. Dispositivos (Se houver): Descrever presença e posicionamento de acessos, cateteres, tubos ou marca-passos.
6. Descrição das Alterações: Se houver achados anormais, descrever o tipo de lesão e a localização exata (ex: "opacidade em terço inferior do pulmão direito").
"""

system_report_format ="""
Você é um médico especialista no setor de Radiologia e seu objetivo é fornecer um laudo médico seguindo estritamente esse formato:

Título: 'RADIOGRAFIA DO TÓRAX – PA E PERFIL'
Findings: 
    - No formato de Bullet Points (-).
    - Qualquer aspecto anomalo que foi encontrado durante a análise das imagens deve ser indicado.
    - Caso não encontrar nenhuma anomlia na análise, não mencione.
    - Contudo, caso o aspecto das estruturas abaixo estejam preservadas, indique que está tudo bem de forma explicita:
        1. 'transparência pulmonar' (ex: - Transparência pulmonar preservada.)
        2. 'Seios costofrênicos' (ex: - Seios costofrênicos livres.)
        3. 'Mediastinos' (ex: Mediastino sem alterações.)
        4. 'Estruturas ósseas' (ex: Estruturas ósseas visualizadas sem alterações.)
"""

def convert_to_conversation(sample):
    conversation = [
        {
            "role": "system",
            "content": system_think_instruction
        },
        {
            "role": "system",
            "content": system_report_format
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["pa_image"]},
                {"type": "image", "image": sample["perfil_image"]},
                {"type": "text", "text": "Fornaça o Laudo das imagens abaixo (PA e Perfil) seguindo a linha de pensamento e o formato de laudo determinado."}
            ]
        },{
            "role": "assistant",
            "content": [
                {"type": "text", "text": f"RADIOGRAFIA DO TÓRAX – PA E PERFIL\n\n{sample["report"]}"}
            ]
        }
    ]

    return { "messages": conversation }

print("definindo o tipo de conversa a ser ajustado.")

In [ ]:
## cedule for loading the dataset.
from datasets import load_dataset

DATASET_ID = "guilhermenf/huac_chest_xray_reports_images"

dataset_train = load_dataset(DATASET_ID, split="train")
dataset_test = load_dataset(DATASET_ID, split="test")

print("carregando os datasets.")

In [ ]:
# turning the dataset to conversation dataset
dataset_conversation_train = [convert_to_conversation(sample) for sample in dataset_train]
dataset_conversation_test = [convert_to_conversation(sample) for sample in dataset_test]

print("convertendo o dataset para o formato correto de treinamento.")

print(dataset_conversation_train[0]["messages"])

# Carregando o Modelo

In [ ]:
!CUDA_VISIBLE_DEVICES=0,1 python -c "import torch; print(torch.cuda.device_count())"

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E4B"
MODEL_PROCESSOR_ID = "google/gemma-4-E4B-it"

# Define model init arguments
model_kwargs = dict(
    dtype=torch.float16, # What torch dtype to use, defaults to auto
)

# BitsAndBytesConfig int-4 config
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=model_kwargs["dtype"],
    bnb_4bit_quant_storage=model_kwargs["dtype"],
)

print("configurações de carregamento do modelo definidas.")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    **model_kwargs)
processor = AutoProcessor.from_pretrained(MODEL_PROCESSOR_ID) # load the model tokenizer (it => instruct => chat_template)

print("modelo carregado com sucesso.")

# Configurações do LoRA

In [ ]:
from peft import LoraConfig
from trl import SFTConfig

peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=32,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"], # make sure to save the lm_head and embed_tokens as you train the special tokens
    ensure_weight_tying=True,
)

args = SFTConfig(
    output_dir="gemma-product-description",     # directory to save and repository id
    num_train_epochs=3,                         # number of training epochs
    per_device_train_batch_size=1,              # batch size per device during training
    optim="adamw_torch_fused",                  # use fused adamw optimizer
    logging_steps=5,                            # log every 5 steps
    save_strategy="epoch",                      # save checkpoint every epoch
    eval_strategy="epoch",                      # evaluate checkpoint every epoch
    learning_rate=2e-4,                         # learning rate, based on QLoRA paper
    bf16=True,                                  # use bfloat16 precision
    max_grad_norm=0.3,                          # max gradient norm based on QLoRA paper
    lr_scheduler_type="constant",               # use constant learning rate scheduler
    push_to_hub=True,                           # push model to hub
    hub_model_id = "joaoneto9/gemma4-x-ray-reports-LoRA-adapters", # O nome que aparecerá no HF
    hub_strategy = "end",                       # Sobe tudo apenas ao final do treino
    hub_token = secret_value_0,                 # Opcional se já fez login via CLI
    report_to="tensorboard",                    # report metrics to tensorboard
    dataset_text_field="",                      # need a dummy field for collator
    dataset_kwargs={"skip_prepare_dataset": True}, # important for collator
    remove_unused_columns = False               # important for collator
)
print("configurações de treino foram definidas")

In [ ]:
from trl import SFTTrainer

# Create a data collator to encode text and image pairs
def collate_fn(examples):
    texts = []
    images = []
    for example in examples:
        image_inputs = process_vision_info(example["messages"])
        text = processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False
        )
        texts.append(text.strip())
        images.append(image_inputs)

    # Tokenize the texts and process the images
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    # The labels are the input_ids, and we mask the padding tokens and image tokens in the loss computation
    labels = batch["input_ids"].clone()

    # Mask tokens for not being used in the loss computation
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == processor.tokenizer.boi_token_id] = -100
    labels[labels == processor.tokenizer.image_token_id] = -100
    labels[labels == processor.tokenizer.eoi_token_id] = -100

    batch["labels"] = labels
    return batch

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset_conversation_train,
    eval_dataset=dataset_conversation_test,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn
)

# Treinar o Modelo

In [ ]:
# Start training, the model will be automatically saved to the Hub and the output directory
trainer.train()

# Liberar a memoria

In [ ]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()